## Step 11 - OCR với PP-OCRv5 (drop-in thay Qwen2.5-VL / step 10)

Notebook này thay engine OCR bằng **PaddleOCR PP-OCRv5** (pipeline det → rec).
So với Qwen2.5-VL-3B (~4.66 s/frame trên T4), PP-OCRv5 nhanh hơn **~30–100x**
(~0.03–0.1 s/frame) khi dùng det mobile + rec latin trên GPU.

**Chất lượng**: `latin_PP-OCRv5_mobile_rec` bao trọn bảng chữ cái Latin nên đọc tốt
tiếng Việt có dấu trên banner/TV. Kém hơn Qwen ở chữ overlap / nền cực phức tạp -
nên chạy smoke test + 1 video mẫu để so sánh trước khi chạy cả corpus.

**Input / output y hệt step 6 / step 10**:
```
input  : Keyframes/<batch>/<video_name>/<idx>.webp
output : OCR/<video_name>.json   # { "<frame_id>": "<ocr text>", ... }
         ocr_data.json           # merged flat dict, drop-in cho backend
```

**Tốc độ ước tính**: ~0.05 s/frame trên T4 → 74.8k frames ≈ **1–1.5 giờ** (1 GPU),
~30–45 phút nếu chạy **2 GPU song song** (`NUM_GPUS=2`, giống notebook 10).

**Resumable 2 cấp**: per video (`OCR/<video>.json` đủ frame thì bỏ qua) +
per frame (frame đã có không re-read, checkpoint mỗi `CHECKPOINT_EVERY`).


In [ ]:
# 1) FIX TORCH: paddleocr -> paddlex -> modelscope -> torch. pip nâng cấp torch lên
#    bản cu126 không hợp NCCL cũ của Kaggle (ImportError: undefined symbol:
#    ncclCommShrink). Cài lại torch 2.5.1 cu121 (bundle sẵn nccl) TRƯỚC.
!python -m pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# 2) PaddleOCR 3.x (paddlex) yêu cầu paddle >= 3.0. PyPI KHÔNG có paddlepaddle-gpu
#    3.x -> cài từ index chính thức của PaddlePaddle. Kaggle T4: bản cu118
#    (driver >= 450 là chạy được). Driver >= 550.54.14 thì đổi sang cu126:
#    -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -m pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!python -m pip install -q -U paddleocr

import torch, paddle
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('paddle', paddle.__version__,
      '| cuda compiled:', paddle.is_compiled_with_cuda(),
      '| gpus:', paddle.device.cuda.device_count())


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path


In [ ]:
# === INPUTS - sửa nếu keyframe dataset đặt tên khác =====================
KEYFRAMES_DIR    = '/kaggle/input/aic-keyframes-webp/Keyframes'
OUTPUT_DIR       = './OCR'                 # one <video_name>.json per video (same as step 6)

# === SONG SONG 2 GPU ====================================================
NUM_GPUS         = 2        # số T4 được cấp; đổi thành 1 nếu chỉ có 1 GPU
CONFIG_FILE      = '_ppocr_config.json'    # config dump cho worker
WORKER_SCRIPT    = '_ppocr_worker.py'      # script worker, sinh tự động ở cell sau

# === MODEL PP-OCRv5 =====================================================
# det : PP-OCRv5_server_det (chính xác hơn, chậm ~8x) | PP-OCRv5_mobile_det (nhanh)
# rec : latin_PP-OCRv5_mobile_rec  - Latin (gồm tiếng Việt có dấu) + số
#       nếu không tìm thấy model latin, fallback 'PP-OCRv5_mobile_rec'
# PP-OCRv6 (mới hơn, chính xác + nhanh hơn v5): đổi det/rec sang 'PP-OCRv6_medium_det'/'PP-OCRv6_medium_rec'
DET_MODEL        = 'PP-OCRv5_mobile_det'
REC_MODEL        = 'latin_PP-OCRv5_mobile_rec'

# === TUNING ============================================================
FRAME_STRIDE     = 1        # 1 = mọi keyframe; 2 = mỗi 2 frame (giảm 1/2 cost)
IMG_BATCH        = 8        # ảnh / 1 lần predict() - gộp batch để ăn GPU
REC_BATCH        = 8        # text line / 1 lần recognition
IMAGE_EXTS       = ('.jpg', '.jpeg', '.png', '.webp')
CHECKPOINT_EVERY = 32       # flush JSON mỗi N frame trong 1 video
MIN_CONF         = 0.0      # lọc text line có score < MIN_CONF; 0.0 = giữ hết

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Mỗi worker init engine 1 lần trên GPU riêng → dump config + sinh worker script.
_config = {
    'keyframes_dir': str(Path(KEYFRAMES_DIR)),
    'output_dir': str(Path(OUTPUT_DIR)),
    'frame_stride': FRAME_STRIDE,
    'det_model': DET_MODEL,
    'rec_model': REC_MODEL,
    'image_batch': IMG_BATCH,
    'rec_batch': REC_BATCH,
    'image_exts': list(IMAGE_EXTS),
    'checkpoint_every': CHECKPOINT_EVERY,
    'min_conf': MIN_CONF,
}
with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
    json.dump(_config, f, ensure_ascii=False, indent=2)
print('config ->', CONFIG_FILE)

WORKER_SRC = r'''import argparse
import json
import os
import sys
import time
from pathlib import Path

from paddleocr import PaddleOCR

CONFIG_PATH = '_ppocr_config.json'
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)

KEYFRAMES_DIR    = cfg['keyframes_dir']
OUTPUT_DIR       = cfg['output_dir']
FRAME_STRIDE     = cfg['frame_stride']
DET_MODEL        = cfg['det_model']
REC_MODEL        = cfg['rec_model']
IMG_BATCH        = cfg['image_batch']
REC_BATCH        = cfg['rec_batch']
IMAGE_EXTS       = tuple(cfg['image_exts'])
CHECKPOINT_EVERY = cfg['checkpoint_every']
MIN_CONF         = cfg['min_conf']

os.makedirs(OUTPUT_DIR, exist_ok=True)


def _img_text(entry):
    d = {}
    try:
        if 'res' in entry:
            d = entry['res']
        elif isinstance(entry, dict):
            d = entry
    except Exception:
        d = entry if isinstance(entry, dict) else {}
    if not isinstance(d, dict):
        d = {}
    texts = d.get('rec_texts') or []
    scores = d.get('rec_scores') or []
    out = []
    for t, s in zip(texts, scores):
        t = (t or '').strip()
        if t and (s is None or float(s) >= MIN_CONF):
            out.append(t)
    return ' '.join(out)


def init_engine(device):
    ocr = PaddleOCR(
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        text_detection_model_name=DET_MODEL,
        text_recognition_model_name=REC_MODEL,
        text_recognition_batch_size=REC_BATCH,
        device=device,
    )
    return ocr


def ocr_paths(paths, ocr):
    result = list(ocr.predict(input=[str(p) for p in paths]))
    return [_img_text(r) for r in result]


def ocr_one(path, ocr):
    return ocr_paths([path], ocr)[0]


def out_path(video_name):
    return Path(OUTPUT_DIR) / f'{video_name}.json'


def load_video_ocr(video_name):
    p = out_path(video_name)
    if p.exists():
        with p.open(encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_video_ocr(video_name, data):
    with out_path(video_name).open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)


def build_video_index():
    root = Path(KEYFRAMES_DIR)
    videos = {}
    for img in root.rglob('*'):
        if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
            video_name = img.parent.name
            frame_id = f'{video_name}_{img.stem}'
            videos.setdefault(video_name, []).append((frame_id, img))
    for v in videos:
        videos[v].sort(key=lambda t: t[0])
    return videos


def run_smoke(device):
    first = next(
        p for p in sorted(Path(KEYFRAMES_DIR).rglob('*'))
        if p.suffix.lower() in IMAGE_EXTS
    )
    print('smoke frame:', first, flush=True)
    ocr = init_engine(device)
    t0 = time.time()
    text = ocr_one(first, ocr)
    print(f'inference: {time.time() - t0:.3f}s', flush=True)
    print('OCR ->', repr(text[:200]), flush=True)


def run_corpus(device, video_names):
    ocr = init_engine(device)
    print(f'[{device}] engine ready: det={DET_MODEL} rec={REC_MODEL}', flush=True)
    videos = build_video_index()
    total_done = 0
    t_start = time.time()
    for video_name in video_names:
        frames = videos[video_name][::FRAME_STRIDE]
        data = load_video_ocr(video_name)
        todo = [(fid, p) for fid, p in frames if fid not in data]
        if not todo:
            total_done += len(frames)
            continue
        done_in_video = 0
        for i in range(0, len(todo), IMG_BATCH):
            chunk = todo[i:i + IMG_BATCH]
            paths = [p for _, p in chunk]
            try:
                texts = ocr_paths(paths, ocr)
            except Exception as e:
                print(f'[WARN] batch failed: {type(e).__name__}: {e}', flush=True)
                texts = [ocr_one(p, ocr) for _, p in chunk]
            for (fid, _p), t in zip(chunk, texts):
                data[fid] = t
            done_in_video += len(chunk)
            if done_in_video % CHECKPOINT_EVERY == 0:
                save_video_ocr(video_name, data)
        save_video_ocr(video_name, data)
        total_done += len(frames)
        print(
            f'[{device}] {video_name}: {len(frames)} frames | '
            f'cum {total_done} | {time.time() - t_start:.1f}s',
            flush=True,
        )
    print(
        f'[{device}] done {total_done} frames in {(time.time() - t_start) / 60:.1f} min',
        flush=True,
    )


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--gpu', type=int, default=-1)
    ap.add_argument('--videos', nargs='*')
    ap.add_argument('--smoke', action='store_true')
    args = ap.parse_args()
    device = 'cpu' if args.gpu < 0 else f'gpu:{args.gpu}'
    if args.smoke:
        run_smoke(device)
    else:
        run_corpus(device, sorted(args.videos))


if __name__ == '__main__':
    main()
'''
with open(WORKER_SCRIPT, 'w', encoding='utf-8') as f:
    f.write(WORKER_SRC)
print('worker script ->', WORKER_SCRIPT)


In [ ]:
# Smoke test qua worker trên GPU 0 - sanity check trước khi chạy cả corpus.
# (Lần đầu sẽ download model det+rec vài MB, hơi lâu.)
cmd = [sys.executable, WORKER_SCRIPT, '--gpu', '0', '--smoke']
r = subprocess.run(cmd, capture_output=True, text=True,
                   encoding='utf-8', errors='replace')
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit('smoke test failed')


In [ ]:
# Group mọi keyframe theo video: {video_name: [(frame_id, path), ...]}
root = Path(KEYFRAMES_DIR)
videos = {}
for img in root.rglob('*'):
    if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
        video_name = img.parent.name                  # e.g. L21_V001
        frame_id   = f'{video_name}_{img.stem}'       # img.stem = zero-padded idx
        videos.setdefault(video_name, []).append((frame_id, img))
for v in videos:
    videos[v].sort(key=lambda t: t[0])

print('videos:', len(videos),
      '| frames:', sum(len(v) for v in videos.values()),
      '| stride:', FRAME_STRIDE)


In [ ]:
# Chia video round-robin cho NUM_GPUS worker, chạy 2 tiến trình song song.
video_names = sorted(videos)
chunks = [video_names[i::NUM_GPUS] for i in range(NUM_GPUS)]
for g in range(NUM_GPUS):
    print(f'gpu{g}: {len(chunks[g])} videos')

t_start = time.time()
procs = []
for gpu, vlist in enumerate(chunks):
    if not vlist:
        continue
    cmd = [sys.executable, WORKER_SCRIPT, '--gpu', str(gpu), '--videos', *vlist]
    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        encoding='utf-8',
        errors='replace',
    )
    procs.append(p)

# Stream output 2 worker live rồi đợi cả 2 xong.
for p in procs:
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(f'worker exited with code {rc}')
elapsed = time.time() - t_start
print(f'\nall workers done in {elapsed/60:.1f} min')


In [ ]:
# Merge các file per-video thành 1 file flat đúng format backend.
# (Khi copy về local: file này sẽ thay thẳng backend/data/ocr/ocr_data.json.)
merged = {}
for jf in sorted(Path(OUTPUT_DIR).glob('*.json')):
    with jf.open(encoding='utf-8') as f:
        merged.update(json.load(f))

with open('ocr_data.json', 'w', encoding='utf-8') as f:
    json.dump(merged, f, ensure_ascii=False)

print(f'merged entries: {len(merged)} -> ocr_data.json')
print(f'non-empty:      {sum(1 for v in merged.values() if v)} '
      f'({sum(1 for v in merged.values() if v)/max(len(merged),1)*100:.1f}%)')


## Dùng với backend

Sau khi notebook chạy xong, download `ocr_data.json` về và copy đúng chỗ:

```bash
# 1) OCR mới thay thẳng file Qwen/EasyOCR cũ (drop-in, format giống hệt)
cp ocr_data.json  backend/data/ocr/ocr_data.json

# 2) Rebuild lại các index phụ thuộc OCR (scene text trong graph cũng đổi):
cd data_processing
python 07_build_text_indexes.py     # ocr_text + asr_text + caption_text index
python 08_build_graph.py            # scene.ocr lấy từ OCR mới
```

## So sánh nhanh trên T4 (1 frame)

| | EasyOCR | Qwen2.5-VL-3B | PP-OCRv5 (mobile det + latin rec) |
|---|---|---|---|
| Tốc độ | ~0.3 s | ~4.66 s | ~0.03–0.1 s |
| Tiếng Việt có dấu | sai dấu nặng | tốt nhất | tốt (latin) |
| Banner nền phức tạp / overlap | hay miss | tốt nhất | khá |
| VRAM | ~1 GB | ~7 GB | <0.5 GB |

## Nâng cấp tuỳ chọn

1. **PP-OCRv6** (mới hơn, chính xác + nhanh hơn v5, 1 model Latin/50 ngôn ngữ):
   đổi `DET_MODEL`/`REC_MODEL` sang `PP-OCRv6_medium_det`/`PP-OCRv6_medium_rec`.
2. **det server cho chất lượng cao hơn**: đổi `DET_MODEL` sang `PP-OCRv5_server_det`
   (chậm ~8x nhưng vẫn ~0.3 s/frame).
3. **Tăng `IMG_BATCH`** (thử 16/32) nếu GPU còn dư VRAM → giảm overhead Python.
4. Nếu vẫn cần chất lượng Qwen cho scene khó, chạy song song: PP-OCRv5 để phủ nhanh,
   Qwen chỉ cho các frame PP-OCRv5 cho kết quả rỗng/đáng ngờ.
